In [ ]:
# !pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.1/en_ner_bc5cdr_md-0.5.1.tar.gz

In [7]:
import pandas as pd
import numpy as np
import scispacy
import spacy
from spacy.matcher import Matcher
from spacy.util import filter_spans
import os
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
nlp = spacy.load("en_ner_bc5cdr_md")

In [4]:
base_url = 'https://raw.githubusercontent.com/londonaicentre/datatakehome/refs/heads/main/data'
notes = pd.read_csv(os.path.join(base_url,'clinical_notes.csv'))

In [5]:

def ner_pc(text):

    doc = nlp(text)

    patterns = [
    [{"LEMMA": {"IN": ["cc", "complaint", "present"]}}, {"IS_PUNCT": True, "OP": "+"},
            {"ENT_TYPE": "DISEASE"},{"LOWER": {"IN":["and"]}, "OP": "?"}, {"ENT_TYPE": "DISEASE", "OP": "?"},],
    [{"LEMMA": {"IN": ["cc", "complaint", "present"]}}, {"IS_ALPHA": True, "OP": "+"},
            {"ENT_TYPE": "DISEASE"},{"LOWER": {"IN":["and"]}, "OP": "?"}, {"ENT_TYPE": "DISEASE", "OP": "?"}, {"IS_PUNCT": True, "OP": "?"}],
    [{"LEMMA": {"IN": ["cc", "complaint", "present"]}}, {"IS_PUNCT": True, "OP": "+"}, {"IS_ALPHA": True, "OP": "{1,3}"}],
    [{"LEMMA": {"IN": ["cc", "complaint", "present"]}}, {"IS_ALPHA": True, "OP": "+"}, {"LOWER": {"IN":["x","since"]}}],
    [{"LEMMA": {"IN": ["pmh", "history"]}}, {"IS_PUNCT": True}, {"ENT_TYPE": "DISEASE"}],
    [{"LEMMA": {"IN": ["pmh", "history"]}},{"IS_ALPHA": True, "OP": "{1,3}"}, {"ENT_TYPE": "DISEASE"}],
    [{"ENT_TYPE": "DISEASE"},{"IS_PUNCT": True, "LEMMA": {"NOT_IN":[":"]}}],
    [{"IS_ALPHA": True, "OP": "{1,4}"}, {"IS_PUNCT": True, "OP": "+", "TEXT": {"NOT_IN": [":"]}}]
            ]

    matcher = Matcher(nlp.vocab)
    matcher.add("PC", patterns)

    matches = matcher(doc)

    text_matches = [doc[start:end] for match_id, start, end in matches]
    filter_text_matches = filter_spans(text_matches)

    text_matches_cleaned = []
    for text in filter_text_matches:
        if 'pmh' not in str(text).lower() and 'history' not in str(text).lower():
            text = str(text).lower()
            text = text.replace('cc:','').replace('complaint','')\
                        .replace(':','').replace('.','')
            text = text.strip()
            text_matches_cleaned.append(text)

    disease_ent = []
    for text in text_matches_cleaned:
        doc2 = nlp(text)
        for ent in doc2.ents:
            # print(ent.text, ent.label_)
            if ent.label_ == 'DISEASE':
                disease_ent.append(ent.text.lower())

    # print(text_matches)
    # print(text_matches_cleaned)
    # print(set(disease_ent))

    if len(disease_ent) > 0:
        return [*set(disease_ent)][0]
    if len(text_matches_cleaned) > 0:
        return text_matches_cleaned[0]
    else:
        return np.nan


In [8]:
notes['NER_PC'] = notes['NOTE_TEXT'].apply(ner_pc)

In [9]:
notes[['NOTE_TEXT','NER_PC']].head(30)

,NOTE_TEXT,NER_PC
0,CC: chest pain\nHPI: 73 year old female with P...,chest pain
1,CC: headache\nHPI: 64 year old female with PMH...,headache
2,CC: palpitations\nHPI: 89 year old male with P...,palpitations
3,6yo male presenting with abdominal pain x sinc...,abdominal pain
4,107yo male presenting with headache x since ye...,chest pain
5,Pt presents to ED with fever. Onset 2 days ago...,fever
6,chest pain. 3 days. Tried over-the-counter ant...,chest pain
7,ED Triage Note\nChief Complaint: rash\nDuratio...,rash
8,cough. 3 days. Tried ibuprofen without relief....,cough
9,Pt presents to ED with chest pain. Onset 1 wee...,chest pain
